In [1]:
import os
import pandas as pd

In [8]:
# Load cleaned annotation-level dataset

mhs = pd.read_csv("../data/processed/mhs_annotations_clean.csv")

print("Loaded shape:", mhs.shape)
mhs.head()

Loaded shape: (135556, 47)


,comment_id,annotator_id,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,...,target_gender_women,target_gender_other,target_origin,target_origin_immigrant,target_origin_migrant_worker,target_origin_specific_country,target_origin_undocumented,target_origin_other,text_original,text_clean
0,47777,10873,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,False,False,False,False,False,False,False,False,Yes indeed. She sort of reminds me of the elde...,Yes indeed. She sort of reminds me of the elde...
1,39773,2790,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,False,False,False,False,False,False,False,False,The trans women reading this tweet right now i...,The trans women reading this tweet right now i...
2,47101,3379,4.0,4.0,4.0,4.0,4.0,4.0,0.0,0.0,...,False,False,True,True,False,False,False,False,Question: These 4 broads who criticize America...,Question: These 4 broads who criticize America...
3,43625,7365,2.0,3.0,2.0,1.0,2.0,0.0,0.0,0.0,...,False,False,True,False,False,False,True,False,It is about time for all illegals to go back t...,It is about time for all illegals to go back t...
4,12538,488,4.0,4.0,4.0,4.0,4.0,4.0,4.0,1.0,...,True,False,False,False,False,False,False,False,For starters bend over the one in pink and kic...,For starters bend over the one in pink and kic...


In [9]:
# Helper function for converting boolean / 0-1 columns safely

def to_binary(series):
    return series.astype(str).str.lower().map({
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0,
        "1.0": 1,
        "0.0": 0
    }).fillna(0).astype(int)

In [10]:
# Define target and annotator metadata columns

target_gender_cols = [
    "target_gender_men",
    "target_gender_non_binary",
    "target_gender_transgender_men",
    "target_gender_transgender_unspecified",
    "target_gender_transgender_women",
    "target_gender_women",
    "target_gender_other"
]

target_origin_cols = [
    "target_origin_immigrant",
    "target_origin_migrant_worker",
    "target_origin_specific_country",
    "target_origin_undocumented",
    "target_origin_other"
]

annotator_gender_cols = [
    "annotator_gender_men",
    "annotator_gender_women",
    "annotator_gender_non_binary",
    "annotator_gender_prefer_not_to_say",
    "annotator_gender_self_describe"
]

annotator_ideology_cols = [
    "annotator_ideology_extremeley_conservative",
    "annotator_ideology_conservative",
    "annotator_ideology_slightly_conservative",
    "annotator_ideology_neutral",
    "annotator_ideology_slightly_liberal",
    "annotator_ideology_liberal",
    "annotator_ideology_extremeley_liberal",
    "annotator_ideology_no_opinion"
]

In [11]:
# Convert all boolean indicator columns to 0/1

binary_cols = (
    target_gender_cols
    + target_origin_cols
    + annotator_gender_cols
    + annotator_ideology_cols
)

for col in binary_cols:
    mhs[col] = to_binary(mhs[col])

print("Converted binary columns.")

Converted binary columns.


In [12]:
# Create target flags

mhs["is_women_targeted"] = mhs["target_gender_women"]

main_immigrant_cols = [
    "target_origin_immigrant",
    "target_origin_migrant_worker",
    "target_origin_specific_country",
    "target_origin_undocumented"
]

mhs["is_immigrant_targeted"] = (
    mhs[main_immigrant_cols]
    .eq(1)
    .any(axis=1)
    .astype(int)
)

print("Women-targeted annotation rows:", mhs["is_women_targeted"].sum())
print("Immigrant-targeted annotation rows:", mhs["is_immigrant_targeted"].sum())

print("Women-targeted unique comments:", mhs.loc[mhs["is_women_targeted"] == 1, "comment_id"].nunique())
print("Immigrant-targeted unique comments:", mhs.loc[mhs["is_immigrant_targeted"] == 1, "comment_id"].nunique())

Women-targeted annotation rows: 27889
Immigrant-targeted annotation rows: 22528
Women-targeted unique comments: 11983
Immigrant-targeted unique comments: 8621


In [13]:
# Create annotator gender group

def get_gender_group(row):
    if row["annotator_gender_women"] == 1:
        return "women"
    elif row["annotator_gender_men"] == 1:
        return "men"
    elif row["annotator_gender_non_binary"] == 1:
        return "non_binary"
    elif row["annotator_gender_prefer_not_to_say"] == 1:
        return "prefer_not_to_say"
    elif row["annotator_gender_self_describe"] == 1:
        return "self_describe"
    else:
        return "unknown"

mhs["annotator_gender_group"] = mhs.apply(get_gender_group, axis=1)

mhs["annotator_gender_group"].value_counts(dropna=False)

annotator_gender_group
women                76370
men                  57582
non_binary             985
prefer_not_to_say      500
self_describe          119
Name: count, dtype: int64

In [14]:
# Create annotator ideology group

def get_ideology_group(row):
    if row["annotator_ideology_extremeley_conservative"] == 1:
        return "extremely_conservative"
    elif row["annotator_ideology_conservative"] == 1:
        return "conservative"
    elif row["annotator_ideology_slightly_conservative"] == 1:
        return "slightly_conservative"
    elif row["annotator_ideology_neutral"] == 1:
        return "neutral"
    elif row["annotator_ideology_slightly_liberal"] == 1:
        return "slightly_liberal"
    elif row["annotator_ideology_liberal"] == 1:
        return "liberal"
    elif row["annotator_ideology_extremeley_liberal"] == 1:
        return "extremely_liberal"
    elif row["annotator_ideology_no_opinion"] == 1:
        return "no_opinion"
    else:
        return "unknown"

mhs["annotator_ideology_group"] = mhs.apply(get_ideology_group, axis=1)

mhs["annotator_ideology_group"].value_counts(dropna=False)

annotator_ideology_group
liberal                   33812
neutral                   23112
slightly_liberal          21333
unknown                   17971
conservative              15628
slightly_conservative     15101
extremely_conservative     4544
no_opinion                 4055
Name: count, dtype: int64

In [15]:
# Create target type label

def get_target_type(row):
    women = row["is_women_targeted"] == 1
    immigrant = row["is_immigrant_targeted"] == 1

    if women and immigrant:
        return "women_and_immigrant"
    elif women:
        return "women_only"
    elif immigrant:
        return "immigrant_only"
    else:
        return "other"

mhs["target_type"] = mhs.apply(get_target_type, axis=1)

mhs["target_type"].value_counts(dropna=False)

target_type
other                  86123
women_only             26905
immigrant_only         21544
women_and_immigrant      984
Name: count, dtype: int64

In [16]:
# Sanity check: subgroup counts for women-targeted content

women_targeted = mhs[mhs["is_women_targeted"] == 1]

women_gender_summary = (
    women_targeted
    .groupby("annotator_gender_group")
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
    .sort_values("annotation_rows", ascending=False)
)

women_gender_summary

,annotator_gender_group,annotation_rows,unique_comments,unique_annotators
4,women,15703,8630,4136
0,men,11854,7168,3169
1,non_binary,209,175,58
2,prefer_not_to_say,92,87,28
3,self_describe,31,31,6


In [17]:
# Sanity check: subgroup counts for immigrant-targeted content

immigrant_targeted = mhs[mhs["is_immigrant_targeted"] == 1]

immigrant_ideology_summary = (
    immigrant_targeted
    .groupby("annotator_ideology_group")
    .agg(
        annotation_rows=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        unique_annotators=("annotator_id", "nunique")
    )
    .reset_index()
    .sort_values("annotation_rows", ascending=False)
)

immigrant_ideology_summary

,annotator_ideology_group,annotation_rows,unique_comments,unique_annotators
2,liberal,5659,3297,1796
3,neutral,3790,2325,1226
6,slightly_liberal,3602,2220,1132
7,unknown,3086,1913,969
0,conservative,2502,1608,805
5,slightly_conservative,2498,1598,776
1,extremely_conservative,762,518,244
4,no_opinion,629,435,219


In [18]:
# Save subgroup audit summaries

os.makedirs("../results/tables", exist_ok=True)

women_gender_summary.to_csv(
    "../results/tables/women_targeted_gender_subgroup_summary.csv",
    index=False
)

immigrant_ideology_summary.to_csv(
    "../results/tables/immigrant_targeted_ideology_subgroup_summary.csv",
    index=False
)

print("Saved subgroup audit summaries.")

Saved subgroup audit summaries.


In [19]:
# Save one master annotation-level file with subgroup flags

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/mhs_annotations_with_subgroups.csv"

mhs.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Final shape:", mhs.shape)

Saved: ../data/processed/mhs_annotations_with_subgroups.csv
Final shape: (135556, 52)


In [20]:
# Final sanity check

final_cols_to_view = [
    "comment_id",
    "annotator_id",
    "text_clean",
    "is_women_targeted",
    "is_immigrant_targeted",
    "target_type",
    "annotator_gender_group",
    "annotator_ideology_group",
    "sentiment",
    "respect",
    "insult",
    "humiliate",
    "status",
    "dehumanize",
    "violence",
    "genocide",
    "attack_defend",
    "hatespeech"
]

mhs[final_cols_to_view].head()

,comment_id,annotator_id,text_clean,is_women_targeted,is_immigrant_targeted,target_type,annotator_gender_group,annotator_ideology_group,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,attack_defend,hatespeech
0,47777,10873,Yes indeed. She sort of reminds me of the elde...,0,0,other,men,neutral,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
1,39773,2790,The trans women reading this tweet right now i...,0,0,other,women,neutral,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.0,0.0
2,47101,3379,Question: These 4 broads who criticize America...,0,1,immigrant_only,men,slightly_conservative,4.0,4.0,4.0,4.0,4.0,4.0,0.0,0.0,4.0,2.0
3,43625,7365,It is about time for all illegals to go back t...,0,1,immigrant_only,men,neutral,2.0,3.0,2.0,1.0,2.0,0.0,0.0,0.0,3.0,0.0
4,12538,488,For starters bend over the one in pink and kic...,1,0,women_only,women,neutral,4.0,4.0,4.0,4.0,4.0,4.0,4.0,1.0,3.0,2.0


In [ ]:
#dropping "other" target type rows to create a main experiment dataset with only clear target groups
main_experiment_df = mhs[mhs["target_type"] != "other"].copy()

main_experiment_df.to_csv(
    "../data/processed/mhs_main_experiment_annotations.csv",
    index=False
)

print("Main experiment rows:", main_experiment_df.shape)

Main experiment rows: (49433, 52)
